# Tech Stock Data Cleaning

This notebook profiles the raw technology stock dataset, validates key data quality checks, applies the same cleaning rules used by `src/data_cleaning.py`, and writes the processed dataset to `data/processed/tech_stocks_clean.csv`.


## 1. Setup

Import the required libraries and define project-relative paths. The path logic works whether the notebook is run from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_cleaning import clean_stock_data

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "tech_stocks.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "tech_stocks_clean.csv"

RAW_DATA_PATH, PROCESSED_DATA_PATH


## 2. Load Raw Data

Load `data/raw/tech_stocks.csv` into a pandas DataFrame for inspection and cleaning.


In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH)
raw_df


## 3. Initial Data Overview

Review the dataset shape, first rows, column data types, missing values, and the count of fully duplicated rows before any cleaning is applied.


In [ ]:
print(f"Shape: {raw_df.shape}")

display(raw_df.head())

display(raw_df.dtypes.rename("dtype").to_frame())

display(raw_df.isna().sum().rename("missing_values").to_frame())

full_duplicate_count = raw_df.duplicated().sum()
print(f"Duplicate row count: {full_duplicate_count}")


## 4. Parse Date Column

Convert the raw `Date` column to datetime values for validation checks. Invalid dates are coerced to missing values so they can be identified before cleaning.


In [ ]:
profile_df = raw_df.copy()
profile_df["Date"] = pd.to_datetime(profile_df["Date"], errors="coerce")

print(f"Date dtype after parsing: {profile_df['Date'].dtype}")
print(f"Rows with unparseable Date values: {profile_df['Date'].isna().sum()}")

display(profile_df[["Date"]].head())


## 5. Row Count by Symbol

Count records for each stock symbol to confirm coverage across tickers and identify symbols with unexpectedly low or high row counts.


In [ ]:
row_count_by_symbol = (
    profile_df.groupby("Symbol", dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
    .sort_values(["row_count", "Symbol"], ascending=[False, True])
)

row_count_by_symbol


## 6. Date Range by Symbol

Check the minimum and maximum trading date for each symbol to validate the time span available for every ticker.


In [ ]:
date_range_by_symbol = (
    profile_df.groupby("Symbol", dropna=False)["Date"]
    .agg(min_date="min", max_date="max")
    .reset_index()
    .sort_values("Symbol")
)

date_range_by_symbol


## 7. Duplicate Symbol-Date Pairs

Identify duplicate `Symbol` and `Date` combinations. These duplicates are important because each symbol should have at most one record per trading date.


In [ ]:
duplicate_symbol_date_pairs = (
    profile_df.groupby(["Symbol", "Date"], dropna=False)
    .size()
    .rename("duplicate_count")
    .reset_index()
    .query("duplicate_count > 1")
    .sort_values(["Symbol", "Date"])
)

print(f"Duplicate Symbol-Date pair count: {len(duplicate_symbol_date_pairs)}")
duplicate_symbol_date_pairs


## 8. Clean the Dataset

Apply the same cleaning logic as `src/data_cleaning.py`: validate required columns, parse dates, sort by symbol and date, remove duplicate rows, remove duplicate `Symbol`-`Date` pairs, lowercase column names, coerce price and volume fields to numeric values, drop rows missing `date`, `symbol`, or `close`, and add `trading_year` and `trading_month` fields.


In [ ]:
clean_df = clean_stock_data(raw_df)

print(f"Raw rows: {len(raw_df)}")
print(f"Clean rows: {len(clean_df)}")
print(f"Rows removed during cleaning: {len(raw_df) - len(clean_df)}")

display(clean_df.head())
display(clean_df.dtypes.rename("dtype").to_frame())


## 9. Save Processed Data

Write the cleaned dataset to `data/processed/tech_stocks_clean.csv`, creating the processed data directory if needed.


In [ ]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Saved {len(clean_df)} rows to {PROCESSED_DATA_PATH}")


## 10. Post-Cleaning Validation

Run final checks on the cleaned DataFrame and saved CSV to confirm that the processed file was written and that key quality rules hold after cleaning.


In [ ]:
saved_df = pd.read_csv(PROCESSED_DATA_PATH, parse_dates=["date"])

validation_summary = pd.DataFrame(
    {
        "check": [
            "saved_file_exists",
            "saved_row_count",
            "clean_dataframe_row_count",
            "missing_date_count",
            "missing_symbol_count",
            "missing_close_count",
            "duplicate_symbol_date_pairs",
        ],
        "value": [
            PROCESSED_DATA_PATH.exists(),
            len(saved_df),
            len(clean_df),
            saved_df["date"].isna().sum(),
            saved_df["symbol"].isna().sum(),
            saved_df["close"].isna().sum(),
            saved_df.duplicated(subset=["symbol", "date"]).sum(),
        ],
    }
)

validation_summary


## Final Summary

The notebook cleaned and validated the technology stock dataset by:

- Loading the raw CSV from `data/raw/tech_stocks.csv`.
- Reviewing shape, sample rows, data types, missing values, and fully duplicated records.
- Parsing `Date` as datetime for profiling and date-based validation.
- Checking row counts and date ranges by `Symbol`.
- Identifying duplicate `Symbol`-`Date` pairs before cleaning.
- Applying the same cleaning logic as `src/data_cleaning.py`, including required-column validation, date parsing, sorting, duplicate removal, lowercase column names, numeric coercion, removal of rows missing `date`, `symbol`, or `close`, and creation of `trading_year` and `trading_month` fields.
- Saving the cleaned dataset to `data/processed/tech_stocks_clean.csv`.
- Running final validation checks against the cleaned DataFrame and saved file.
